# two-optimizers-alternating-step — ex2: WGAN-style n_critic D-steps per G-step

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `two-optimizers-alternating-step`. Running the final beacon cell reports progress against the `GAN: Two-optimizers alternating step` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: Two-optimizers alternating step` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`two-optimizers-alternating-step`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "two-optimizers-alternating-step"
DD_SUBTOPIC = "GAN: Two-optimizers alternating step"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## GAN: two-optimizers alternating step — deepening refresher

The vanilla pattern is **one D-step + one G-step per iteration**. But in practice the discriminator often needs MORE updates than the generator to stay strong enough to produce useful gradients (WGAN-GP uses `n_critic = 5` D-steps per G-step).

```python
for iteration in range(N):
    # k discriminator updates
    for _ in range(n_critic):
        D_opt.zero_grad()
        z = torch.randn(B, z_dim)
        fake = G(z).detach()             # stop-grad into G
        loss_D = (D(fake) - D(x_real)).mean()
        loss_D.backward()
        D_opt.step()
    # 1 generator update
    G_opt.zero_grad()
    z = torch.randn(B, z_dim)
    fake = G(z)                          # grad flows into G
    loss_G = -D(fake).mean()
    loss_G.backward()
    G_opt.step()
```

**Invariants.** Each D-step uses a FRESH `z` (you can also reuse — both are seen in the wild). The G-step uses its own fresh `z`, NOT detached. Every step zeros only ITS optimizer's grads. `n_critic=1` recovers the Goodfellow alternation.

### Exercise 2 — WGAN-style n_critic D-steps per G-step

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the WGAN-style training pattern of `n_critic` D-step updates followed by 1 G-step update per outer iteration, with fresh noise per inner step.
> Keywords: gan, wgan, n-critic, d-step-loop
> ```

**KCs targeted:** `k-d-steps-per-g-step`, `per-step-fresh-z-sampling`

Implement `ex2_wgan_iter(G, D, G_opt, D_opt, x_real, z_dim, n_critic)`.

One outer iteration of the WGAN training loop:

**Inner loop — `n_critic` D-step updates:**
  For each of the `n_critic` inner steps:
    a. `D_opt.zero_grad()`
    b. Sample FRESH `z = t.randn(B, z_dim)` (NOT reuse across steps).
    c. `fake = G(z).detach()` — stop-grad into G.
    d. `loss_D = (D(fake) - D(x_real)).mean()` (simplified Wasserstein-style).
    e. `loss_D.backward()`
    f. `D_opt.step()`
    g. Record `loss_D.item()` in a list.

**Outer — 1 G-step update:**
  a. `G_opt.zero_grad()`
  b. Sample FRESH `z` (independent of the inner-loop noises).
  c. `fake = G(z)` — NO detach, grad flows into G.
  d. `loss_G = -D(fake).mean()`
  e. `loss_G.backward()`
  f. `G_opt.step()`

Return `(d_losses, loss_G_value)` — a Python list of `n_critic` floats and a single float.

Infer `B = x_real.shape[0]` from the input.

Inputs:
- `G`, `D`: `nn.Module`s.
- `G_opt`, `D_opt`: optimizers wired to G.parameters() / D.parameters() respectively.
- `x_real`: real samples, shape `(B, x_dim)`.
- `z_dim`: noise dimension (int).
- `n_critic`: number of D-step updates per G-step (int, >=1).

Output: `(list[float], float)`.

In [ ]:
def ex2_wgan_iter(G, D, G_opt, D_opt, x_real, z_dim, n_critic):
    """One outer WGAN iteration: n_critic D-steps + 1 G-step. Return (d_losses, loss_G)."""
    raise NotImplementedError()


def _test_ex2():
    import torch.nn as nn

    t.manual_seed(0)
    z_dim, x_dim, B = 4, 6, 8
    G = nn.Sequential(nn.Linear(z_dim, 16), nn.ReLU(), nn.Linear(16, x_dim))
    D = nn.Sequential(nn.Linear(x_dim, 16), nn.ReLU(), nn.Linear(16, 1))
    G_opt = t.optim.SGD(G.parameters(), lr=1e-2)
    D_opt = t.optim.SGD(D.parameters(), lr=1e-2)
    x_real = t.randn(B, x_dim)

    n_critic = 5
    G_before = {n: p.detach().clone() for n, p in G.named_parameters()}
    D_before = {n: p.detach().clone() for n, p in D.named_parameters()}

    d_losses, loss_G_val = ex2_wgan_iter(G, D, G_opt, D_opt, x_real, z_dim, n_critic)

    # Return shape.
    assert isinstance(d_losses, list), f'd_losses must be list, got {type(d_losses).__name__}'
    assert len(d_losses) == n_critic, (
        f'expected {n_critic} D-step losses, got {len(d_losses)}'
    )
    for dl in d_losses:
        assert isinstance(dl, float), f'each D-loss must be float, got {type(dl).__name__}'
        assert t.isfinite(t.tensor(dl)).item(), f'D-loss non-finite: {dl}'
    assert isinstance(loss_G_val, float), f'loss_G must be float, got {type(loss_G_val).__name__}'
    assert t.isfinite(t.tensor(loss_G_val)).item(), f'G-loss non-finite: {loss_G_val}'

    # Both modules moved.
    G_moved = any(not t.allclose(G_before[n], p.detach()) for n, p in G.named_parameters())
    D_moved = any(not t.allclose(D_before[n], p.detach()) for n, p in D.named_parameters())
    assert G_moved, 'G params did not move — did you call G_opt.step()?'
    assert D_moved, 'D params did not move — did you run the D-step inner loop?'

    # n_critic=1 must collapse to the vanilla pattern (one D-step + one G-step).
    t.manual_seed(1)
    G2 = nn.Sequential(nn.Linear(z_dim, 16), nn.ReLU(), nn.Linear(16, x_dim))
    D2 = nn.Sequential(nn.Linear(x_dim, 16), nn.ReLU(), nn.Linear(16, 1))
    G2_opt = t.optim.SGD(G2.parameters(), lr=1e-2)
    D2_opt = t.optim.SGD(D2.parameters(), lr=1e-2)
    x_real2 = t.randn(B, x_dim)
    d_losses_1, _ = ex2_wgan_iter(G2, D2, G2_opt, D2_opt, x_real2, z_dim, n_critic=1)
    assert len(d_losses_1) == 1, f'n_critic=1 should give exactly 1 D-loss, got {len(d_losses_1)}'

    # Stability across 20 outer iterations with n_critic=3.
    t.manual_seed(2)
    G3 = nn.Sequential(nn.Linear(z_dim, 16), nn.ReLU(), nn.Linear(16, x_dim))
    D3 = nn.Sequential(nn.Linear(x_dim, 16), nn.ReLU(), nn.Linear(16, 1))
    G3_opt = t.optim.SGD(G3.parameters(), lr=1e-3)
    D3_opt = t.optim.SGD(D3.parameters(), lr=1e-3)
    for _ in range(20):
        x_r = t.randn(B, x_dim)
        dls, lg = ex2_wgan_iter(G3, D3, G3_opt, D3_opt, x_r, z_dim, n_critic=3)
        assert len(dls) == 3
        for dl in dls:
            assert t.isfinite(t.tensor(dl)).item()
        assert t.isfinite(t.tensor(lg)).item()
    for p in G3.parameters():
        assert t.isfinite(p).all(), 'G params went non-finite over 20 iters'
    for p in D3.parameters():
        assert t.isfinite(p).all(), 'D params went non-finite over 20 iters'

    # Param isolation between optimizers (no cross-wiring).
    G3_ids = {id(p) for g in G3_opt.param_groups for p in g['params']}
    for p in D3.parameters():
        assert id(p) not in G3_ids, 'CROSS-WIRING: D param in G_opt'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_wgan_iter(G, D, G_opt, D_opt, x_real, z_dim, n_critic):
    B = x_real.shape[0]
    d_losses = []
    # === inner loop: n_critic D-steps ===
    for _ in range(n_critic):
        D_opt.zero_grad()
        z = t.randn(B, z_dim)
        fake = G(z).detach()
        loss_D = (D(fake) - D(x_real)).mean()
        loss_D.backward()
        D_opt.step()
        d_losses.append(loss_D.item())
    # === outer: 1 G-step ===
    G_opt.zero_grad()
    z = t.randn(B, z_dim)
    fake = G(z)
    loss_G = -D(fake).mean()
    loss_G.backward()
    G_opt.step()
    return d_losses, loss_G.item()
```

**Why fresh `z` per inner step.** Each D-step asks D to distinguish a NEW fake from the real data. Reusing the same `z` would let D memorize one particular noise sample's shortcomings — defeats the point of the critic loop, which is to give D a strong signal over the FAKE DISTRIBUTION, not one fake point.

**Why `n_critic` D-steps for WGAN.** The Wasserstein critic needs to be close to optimal at each iteration for the EM distance estimate to be reliable. The original WGAN paper uses 5; WGAN-GP also uses 5. With `n_critic=1` you recover the Goodfellow vanilla alternation — useful for sanity checks (the code path is the same, just with one inner iteration).

**Per-optimizer zero_grad.** `D_opt.zero_grad()` inside the inner loop, `G_opt.zero_grad()` once outside. Calling each optimizer's `zero_grad()` right before its `step()` is the convention that keeps the code readable as you scale up n_critic — never wonder which set of grads is being cleared.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()